# 温度駆動力を与える。固相液相の**多結晶**シミュレーションー＞GPU計算
# ー＞ https://doi.org/10.1016/j.mtla.2023.101702 の再現
# 固液界面のみ異方性ー＞ active parameter trackingで高速化？
# ー＞　異方性の部分の挙動調整

In [ ]:
import os
import math
import numpy as np
import matplotlib.pyplot as plt
from numba import cuda, float32,int32
from scipy.spatial.transform import Rotation

# =========================
# 0) Settings / Parameters
# =========================
nx, ny = 1024, 1024
number_of_grain = 5         # 0: liquid, 1..N-1: solid grains
dx, dy = 2e-5, 2e-5
dt = 0.002
nsteps = 5000             
pi = np.pi

delta = 6.0 * dx
T_melt = 1687.15
G = 1.0e+02
V_pulling = 5.0
Sf =2.12e+04
# ---- Energetic anisotropy (Appendix A / Table values you had) ----
a0 = 54.7 * pi / 180.0
delta_a = 0.36
mu_a = 0.6156
p_round = 0.05

# ---- Interface energies (scale you want) ----
# Solid-Liquid base energy (use (100) baseline)
gamma_100 = 0.44
# Grain boundary energy (isotropic)
gamma_GB = 0.60

# ---- Mobilities (choose reasonable constants) ----
# Solid-liquid mobility (isotropic here; you can later add b(theta))
M_SL = 4.62e-4
# Grain boundary mobility (isotropic)
M_GB = M_SL*0.05
# Output
out_dir = "result/periodic/ani_quat"
os.makedirs(out_dir, exist_ok=True)

In [ ]:
# =====================================
# 1) Grain orientations (quaternions)  [paper-like 2D]
#   - Out-of-plane direction fixed to <110>
#   - Each grain: random rotation around that axis (1 DOF)
# =====================================
np.random.seed(42)
N = number_of_grain

grain_quaternions = np.zeros((N, 4), dtype=np.float64)
grain_quaternions[0] = np.array([0.0, 0.0, 0.0, 1.0])  # liquid dummy (x,y,z,w)

# --- define <110> axis (unit) as "out-of-plane" axis ---
u110 = np.array([1.0, 1.0, 0.0], dtype=np.float64)
u110 /= np.linalg.norm(u110)

# --- build a rotation that aligns global z-axis to <110> ---
z = np.array([0.0, 0.0, 1.0], dtype=np.float64)

axis = np.cross(z, u110)
axis_n = np.linalg.norm(axis)
dot_ = float(np.clip(np.dot(z, u110), -1.0, 1.0))

if axis_n < 1e-12:
    R_align = Rotation.identity()
else:
    axis /= axis_n
    ang = math.acos(dot_)
    R_align = Rotation.from_rotvec(axis * ang)

# --- random twist around <110> for each grain ---
for gid in range(1, N):
    theta = np.random.uniform(0.0, 2.0 * np.pi)  # 0..2pi
    R_twist = Rotation.from_rotvec(u110 * theta)

    # Apply alignment then twist (order can be swapped; this is consistent & stable)
    Rm = R_twist * R_align

    grain_quaternions[gid] = Rm.as_quat()  # (x,y,z,w) SciPy convention


# ---- Precompute rotated {111} normals (8) for each grain ----
n111_base = np.array([
    [ 1,  1,  1], [ 1,  1, -1], [ 1, -1,  1], [-1,  1,  1],
    [ 1, -1, -1], [-1,  1, -1], [-1, -1,  1], [-1, -1, -1],
], dtype=np.float32) / np.sqrt(3.0)

grain_n111 = np.zeros((N, 8, 3), dtype=np.float32)
for gid in range(N):
    Rmat = Rotation.from_quat(grain_quaternions[gid]).as_matrix().astype(np.float32)
    grain_n111[gid] = (Rmat @ n111_base.T).T  # (8,3)


In [ ]:

# ==========================================================
# 2) Constant GB interactions (wij,aij,mij)  (i,j>0 constant)
#    and baseline SL constants (for reference only)
# ==========================================================
wij = np.zeros((N, N), dtype=np.float32)
aij = np.zeros((N, N), dtype=np.float32)
mij = np.zeros((N, N), dtype=np.float32)

In [ ]:
# --- helper conversions ---
def eps_from_gamma(gamma):
    # epsilon = sqrt(8*delta*gamma/pi^2)
    return math.sqrt(8.0 * delta * gamma / (pi * pi))

def w_from_gamma(gamma):
    # w = 4*gamma/delta
    return 4.0 * gamma / delta

def mij_from_M(M):
    # mij(phi) = (pi^2/(8*delta)) * M   (same form you used)
    return (pi * pi / (8.0 * delta)) * M

# ---- GB constants (isotropic) ----
eps_GB = eps_from_gamma(gamma_GB)
w_GB = w_from_gamma(gamma_GB)
m_GB_phi = mij_from_M(M_GB)

for i in range(1, N):
    for j in range(1, N):
        if i == j:
            continue
        wij[i, j] = w_GB
        aij[i, j] = eps_GB
        mij[i, j] = m_GB_phi

# ---- SL baseline (100) ----
eps0_sl = eps_from_gamma(gamma_100)
w0_sl = w_from_gamma(gamma_100)
m_sl_phi = mij_from_M(M_SL)

for i in range(1, N):
    # store baseline (will be overwritten locally in kernel for anisotropy)
    wij[0, i] = wij[i, 0] = w0_sl
    aij[0, i] = aij[i, 0] = eps0_sl
    mij[0, i] = mij[i, 0] = m_sl_phi

In [ ]:
NG = 20

@cuda.jit(device=True, inline=True)
def calc_a_from_cos(cosT, a0, delta_a, mu_a, p_round):
    # Appendix A (A2)-(A4)
    c2 = cosT*cosT
    C = math.sqrt(c2 + p_round*p_round)
    S = math.sqrt(max(1.0 - c2, 0.0) + p_round*p_round)
    return mu_a * (1.0 + delta_a * (C + math.tan(a0)*S))

In [ ]:
@cuda.jit(device=True, inline=True)
def best_cos_from_grad(gx, gy, n111, solidid, g2_floor):
    g = math.sqrt(gx*gx + gy*gy) 
    if g < g2_floor:
        return 0.0

    nxn = gx / g
    nyn = gy / g

    best = 0.0
    for t in range(8):
        n111x = n111[solidid, t, 0]
        n111y = n111[solidid, t, 1]
        c = nxn*n111x + nyn*n111y  # 液相のフェーズフィールド変数と（111）の内積
        ac = abs(c)
        if ac > best:
            best = ac

    if best < 0.0: best = 0.0
    if best > 1.0: best = 1.0
    return best


In [ ]:
from numba import cuda, float32
import math

# -----------------------------------------------------
# helper: x periodic, y clamp (same as kernel)
# -----------------------------------------------------
@cuda.jit(device=True, inline=True)
def idx_xp(l, nx):
    return l + 1 if l < nx - 1 else 0

@cuda.jit(device=True, inline=True)
def idx_xm(l, nx):
    return l - 1 if l > 0 else nx - 1

@cuda.jit(device=True, inline=True)
def idx_yp(m, ny):
    # Neumann: top is mirror (m stays)
    return m + 1 if m < ny - 1 else m

@cuda.jit(device=True, inline=True)
def idx_ym(m, ny):
    # Neumann: bottom is mirror (m stays at 0)
    return m - 1 if m > 0 else 0

@cuda.jit(device=True, inline=True)
def grad_phi_xy(phi, gid, l, m, nx, ny, dx):
    lp = idx_xp(l, nx)
    lm = idx_xm(l, nx)
    mp = idx_yp(m, ny)
    mm = idx_ym(m, ny)
    gx = (phi[gid, lp, m] - phi[gid, lm, m]) / (2 * dx)
    gy = (phi[gid, l, mp] - phi[gid, l, mm]) / (2 * dx)  
    return gx, gy


In [ ]:
@cuda.jit(device=True, inline=True)
def eps2_at_cell_from_liquid(phi, l, m, nx, ny, dx, solidid, eps0_sl, a0, delta_a, mu_a, p_round, n111, g2_floor):
    # --- get gradient of liquid phase field ---
    gx, gy = grad_phi_xy(phi, 0, l, m, nx, ny, dx)

    # --- get best cos(theta) between grad and n111 of this grain ---
    best_cos = best_cos_from_grad(gx, gy, n111, solidid, g2_floor)

    # --- get anisotropic epsilon ---
    a_sl = calc_a_from_cos(best_cos, a0, delta_a, mu_a, p_round)
    eps_sl = eps0_sl * a_sl

    return eps_sl * eps_sl

In [ ]:
@cuda.jit(device=True, inline=True)
def aniso_term1_liquid(phi, l, m, nx, ny, dx, solidid, eps0_sl, a0, delta_a, mu_a, p_round, n111, g2_floor):
    # 境界条件
    lp = idx_xp(l, nx)
    lm = idx_xm(l, nx)
    mp = idx_yp(m, ny)
    mm = idx_ym(m, ny)
    # eps^2 at center cell
    eps2_c = eps2_at_cell_from_liquid(phi, l, m, nx, ny, dx, solidid, eps0_sl, a0, delta_a, mu_a, p_round, n111, g2_floor)
    # eps^2 at neighboring cells
    eps2_xp = eps2_at_cell_from_liquid(phi, lp, m, nx, ny, dx, solidid, eps0_sl, a0, delta_a, mu_a, p_round, n111, g2_floor)
    eps2_xm = eps2_at_cell_from_liquid(phi, lm, m, nx, ny, dx, solidid, eps0_sl, a0, delta_a, mu_a, p_round, n111, g2_floor)
    eps2_yp = eps2_at_cell_from_liquid(phi, l, mp, nx, ny, dx, solidid, eps0_sl, a0, delta_a, mu_a, p_round, n111, g2_floor)
    eps2_ym = eps2_at_cell_from_liquid(phi, l, mm, nx, ny, dx, solidid, eps0_sl, a0, delta_a, mu_a, p_round, n111, g2_floor)
    # 液相のphi
    phi_c = phi[0, l, m]
    phi_xp = phi[0, lp, m]
    phi_xm = phi[0, lm, m]
    phi_yp = phi[0, l, mp]
    phi_ym = phi[0, l, mm]
    # 中心差分で計算
    Fx_p = (eps2_c + eps2_xp) * (phi_xp - phi_c) / (2 * dx)
    Fx_m = (eps2_c + eps2_xm) * (phi_c - phi_xm) / (2 * dx)
    Fy_p = (eps2_c + eps2_yp) * (phi_yp - phi_c) / (2 * dx)
    Fy_m = (eps2_c + eps2_ym) * (phi_c - phi_ym) / (2 * dx)
    
    return (Fx_p - Fx_m) / (2 * dx) + (Fy_p - Fy_m) / (2 * dx)

In [ ]:
@cuda.jit(device=True, inline=True)
def build_active_list(phi, l, m, nx, ny, number_of_grain,threshold, KMAX, out_ids):
    lp = idx_xp(l, nx)
    lm = idx_xm(l, nx)
    mp = idx_yp(m, ny)
    mm = idx_ym(m, ny)

    nf = 0
    out_ids[nf] = 0
    nf += 1

    for i in range(1, number_of_grain):
        c = phi[i, l, m]
        if (c > threshold or phi[i, lp, m] > threshold or phi[i, lm, m] > threshold or phi[i, l, mp] > threshold or phi[i, l, mm] > threshold):
            if nf < KMAX:
                out_ids[nf] = i
                nf += 1
    return nf

In [ ]:
from numba import cuda, float32, int32
import math

LIQ = 0
KMAX = 15

@cuda.jit
def kernel_update_phasefield_active(phi, phi_new, temp,
                                    wij, aij, mij, n111,
                                    nx, ny, number_of_grain,
                                    dx, dt, T_melt, Sf,
                                    eps0_sl, w0_sl,
                                    a0, delta_a, mu_a, p_round,
                                    threshold, g2_floor):

    l, m = cuda.grid(2)
    if l >= nx or m >= ny:
        return

    # active list
    active = cuda.local.array(KMAX, int32)
    nf = build_active_list(phi, l, m, nx, ny, number_of_grain, threshold, KMAX, active)

    if nf <= 1:
        # 念のため：全相コピー（または液相=1）
        for i in range(number_of_grain):
            phi_new[i, l, m] = phi[i, l, m]
        return

    lp = idx_xp(l, nx); lm = idx_xm(l, nx)
    mp = idx_yp(m, ny); mm = idx_ym(m, ny)
    inv_dx2 = 1.0 / (dx * dx)

    Tcur = temp[l, m]

    # 固液 anisotropic term1 を active固相ごとにキャッシュ
    lap_sl = cuda.local.array(KMAX, float32)
    for t in range(nf):
        gid = active[t]
        if gid == LIQ:
            lap_sl[t] = 0.0
        else:
            lap_sl[t] = aniso_term1_liquid(phi, l, m, nx, ny, dx,
                                           gid, eps0_sl,
                                           a0, delta_a, mu_a, p_round,
                                           n111, g2_floor)

    # w_sl（君の方式：支配固相 i_s の方位で決める）
    i_s = 1
    maxv = 0.0
    for t in range(nf):
        gid = active[t]
        if gid == LIQ:
            continue
        v = phi[gid, l, m]
        if v > maxv:
            maxv = v
            i_s = gid

    phix = (phi[i_s, lp, m] - phi[i_s, lm, m]) * (0.5 / dx)
    phiy = (phi[i_s, l, mp] - phi[i_s, l, mm]) * (0.5 / dx)
    gn2 = phix*phix + phiy*phiy

    cmax = 0.0
    if gn2 >= g2_floor:
        inv_gn = 1.0 / math.sqrt(gn2)
        nx_ = phix * inv_gn
        ny_ = phiy * inv_gn
        for tt in range(8):
            c = abs(nx_*n111[i_s, tt, 0] + ny_*n111[i_s, tt, 1])
            if c > cmax:
                cmax = c
        if cmax > 1.0:
            cmax = 1.0

    a_loc = calc_a_from_cos(cmax, a0, delta_a, mu_a, p_round)
    w_sl = w0_sl * (a_loc * a_loc)

    # phi_new をゼロ（slice不可なのでループ）
    for i in range(number_of_grain):
        phi_new[i, l, m] = 0.0

    # update active only
    for t1 in range(nf):
        i = active[t1]
        dpi = 0.0

        for t2 in range(nf):
            j = active[t2]
            if i == j:
                continue

            # driving force (solid-liquid only)
            driving_force = 0.0
            if (i != LIQ and j == LIQ):
                driving_force = -Sf * (Tcur - T_melt)
            elif (i == LIQ and j != LIQ):
                driving_force =  Sf * (Tcur - T_melt)

            ppp = 0.0
            for t3 in range(nf):
                k = active[t3]

                lap_k = (phi[k, lp, m] + phi[k, lm, m] +
                         phi[k, l, mp] + phi[k, l, mm] -
                         4.0 * phi[k, l, m]) * inv_dx2

                # potential term W
                wik = wij[i, k]
                wjk = wij[j, k]
                if k == LIQ:
                    if i != LIQ: wik = w_sl
                    if j != LIQ: wjk = w_sl
                term1 = (wik - wjk) * phi[k, l, m]

                # gradient term
                term2 = 0.0
                if (j == LIQ and i != LIQ):
                    if k == i:
                        term2 = 0.5 * lap_sl[t1]
                elif (i == LIQ and j != LIQ):
                    if k == j:
                        term2 = -0.5 * lap_sl[t2]
                else:
                    eps2ik = aij[i, k] * aij[i, k]
                    eps2jk = aij[j, k] * aij[j, k]
                    term2 = 0.5 * (eps2ik - eps2jk) * lap_k

                ppp += term1 + term2

            # force term（mijはここに入れない）
            phii_phij = phi[i, l, m] * phi[j, l, m]
            term_force = (8.0 / 3.1415926535) * math.sqrt(max(phii_phij, 0.0)) * driving_force

            dpi -= 2.0 * mij[i, j] / float(nf) * (ppp - term_force)

        phi_new[i, l, m] = phi[i, l, m] + dt * dpi

    # projection on active only
    s = 0.0
    for t1 in range(nf):
        i = active[t1]
        v = phi_new[i, l, m]
        if v < 0.0: v = 0.0
        if v > 1.0: v = 1.0
        phi_new[i, l, m] = v
        s += v

    if s > 1e-20:
        invs = 1.0 / s
        for t1 in range(nf):
            i = active[t1]
            phi_new[i, l, m] *= invs
    else:
        phi_new[LIQ, l, m] = 1.0

In [ ]:
@cuda.jit
def kernel_update_temp(temp, cooling_rate, nx, ny):
    l, m = cuda.grid(2)
    if l < nx and m < ny:
        temp[l, m] -= cooling_rate

In [ ]:
# =====================================================
# 4) Initialization
# =====================================================
phi_cpu = np.zeros((number_of_grain, nx, ny), dtype=np.float32)
seed_height = 128
factor = 2.2 / delta

n_solid = number_of_grain - 1
grain_width = nx // n_solid

for l in range(nx):
    grain_id = int(l // grain_width) + 1
    if grain_id > n_solid:
        grain_id = n_solid
    for m in range(ny):
        y = m * dy
        dist = y - (seed_height * dy)
        phi_solid = 0.5 * (1.0 - np.tanh(factor * dist))
        phi_cpu[grain_id, l, m] = phi_solid
        phi_cpu[0, l, m] = 1.0 - phi_solid

temp_cpu = np.zeros((nx, ny), dtype=np.float64)
for m in range(ny):
    temp_cpu[:, m] = T_melt + G * (m - seed_height) * dy

# quick plot step0
phase_map = np.argmax(phi_cpu, axis=0)
plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.imshow(phase_map.T, cmap="tab20", origin="lower", interpolation="nearest")
plt.colorbar(ticks=range(number_of_grain), label="Phase ID")
plt.title("Initial Grain Map")

plt.subplot(1, 3, 2)
plt.imshow(temp_cpu.T, cmap="hot", origin="lower")
plt.colorbar(label="T [K]")
plt.title("Initial Temperature")

plt.subplot(1, 3, 3)
plt.imshow(phi_cpu[1].T, cmap="bwr", origin="lower", vmin=0, vmax=1)
plt.colorbar()
plt.title("phi(grain 1) step0")
plt.tight_layout()
plt.savefig(os.path.join(out_dir, "step_0.png"), dpi=150)
plt.close()

In [ ]:
# dtypeを統一（推奨）
phi_cpu  = phi_cpu.astype(np.float32)
temp_cpu = temp_cpu.astype(np.float32)

d_phi     = cuda.to_device(phi_cpu)
d_phi_new = cuda.to_device(phi_cpu)
d_temp    = cuda.to_device(temp_cpu)
d_wij     = cuda.to_device(wij.astype(np.float32))
d_aij     = cuda.to_device(aij.astype(np.float32))
d_mij     = cuda.to_device(mij.astype(np.float32))
d_n111    = cuda.to_device(grain_n111.astype(np.float32))

threadsperblock = (16, 16)
blockspergrid = (math.ceil(nx / threadsperblock[0]),
                 math.ceil(ny / threadsperblock[1]))

cooling_rate = np.float32(G * V_pulling * dt)  # 温度固定なら0でOK
T_melt_f  = np.float32(T_melt)
Sf_f  = np.float32(Sf)

In [ ]:
# =====================================================
# 6) Main loop
# =====================================================
save_every = 100
threshold_f = np.float32(1e-6)
g2_floor_f = np.float32(1e-9)      # dx=1e-4ならこのへん
for nstep in range(1, nsteps + 1):
    kernel_update_temp[blockspergrid, threadsperblock](d_temp, cooling_rate, nx, ny)

    kernel_update_phasefield_active[blockspergrid, threadsperblock](
        d_phi, d_phi_new, d_temp,
        d_wij, d_aij, d_mij,
        d_n111,
        nx, ny, number_of_grain, np.float32(dx), np.float32(dt),
        T_melt_f, Sf_f,
        np.float32(eps0_sl), np.float32(w0_sl),
        np.float32(a0), np.float32(delta_a), np.float32(mu_a), np.float32(p_round),
        threshold_f, g2_floor_f
    )


    d_phi, d_phi_new = d_phi_new, d_phi

    if nstep % save_every == 0:
        current_phi  = d_phi.copy_to_host()
        phase_map = np.argmax(current_phi, axis=0)
        current_temp = d_temp.copy_to_host()
        print(f"Step {nstep} | Tmin={current_temp.min():.3f} Tmax={current_temp.max():.3f}")

        plt.figure(figsize=(15, 5))
        plt.subplot(1, 3, 1)
        plt.imshow(phase_map.T, cmap="tab20", origin="lower", interpolation="nearest")
        plt.colorbar(ticks=range(number_of_grain), label="Phase ID")
        plt.title(f"Phases step {nstep}")

        plt.subplot(1, 3, 2)
        plt.imshow(current_temp.T, cmap="hot", origin="lower")
        plt.colorbar(label="T [K]")
        plt.title(f"Temperature step {nstep}")

        plt.subplot(1, 3, 3)
        plt.imshow(current_phi[1].T, cmap="bwr", origin="lower", vmin=0, vmax=1)
        plt.colorbar()
        plt.title(f"phi(grain 1) step {nstep}")

        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, f"step_{nstep}.png"), dpi=150)
        plt.close()

